# Getting Started with AkasicDB and LangChain

This notebook builds a small Retrieval-Augmented Generation (RAG) application with LangChain, OpenAI models, and AkasicDB as the vector store.

The flow is intentionally close to the LangChain RAG tutorial:

1. Configure the chat model, embedding model, and AkasicDB connection.
2. Load a web document.
3. Split the document into chunks.
4. Store the chunks and embeddings in AkasicDB.
5. Retrieve relevant chunks for a question.
6. Ask the chat model to answer using only the retrieved context.

AkasicDB handles the vector storage and similarity search, while LangChain gives us reusable building blocks for loading documents, splitting text, prompting, and composing the RAG pipeline.


## Prerequisites

You need the following before running the notebook:

- An OpenAI API key for the chat and embedding models.
- An AkasicDB database instance. (You can create one [here](https://db.akasic.cloud).)
- A database connection string saved as `DB_DSN` in Google Colab Secrets.

In Google Colab, open the key icon in the left sidebar and add:

- `OPENAI_API_KEY`
- `DB_DSN`

This notebook assumes it is running in Google Colab and uses Colab Secrets directly, following the same connection pattern as the basic AkasicDB example.


## Step 1: Install dependencies

These packages cover the LangChain RAG pieces, OpenAI model wrappers, LangGraph orchestration, the AkasicDB vector store integration, and a small HTML fetch/parsing step.

This notebook intentionally avoids `langchain-community` and `langchain-text-splitters` in Colab. Those packages are useful, but Colab's preinstalled packages can occasionally get out of sync with freshly installed LangChain dependencies. For this tutorial, simple `requests`, `BeautifulSoup`, and a small chunking helper keep the example focused on AkasicDB and the RAG pipeline.


In [ ]:
%pip install --quiet --upgrade "langchain[openai]" langchain-core langchain-openai langgraph langchain-akasicdb psycopg requests beautifulsoup4


## Step 2: Configure API keys and database connection

This cell follows the connection setup from `akasicdb_example.ipynb` as closely as possible: import `userdata`, read `DB_DSN` from Colab Secrets, and provide a small `get_conn()` helper.

The only LangChain-specific addition is `DB_URL`, which converts the same DSN to the `postgresql+psycopg://...` form used by SQLAlchemy-based vector store integrations when needed.


In [ ]:
import os

import psycopg
from google.colab import userdata


# Configure OpenAI API Key
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Configure AkasicDB Connection String
DB_DSN = os.getenv("DATABASE_URL", userdata.get("DB_DSN"))


def get_conn(conn_string: str = DB_DSN):
    return psycopg.connect(conn_string)


def to_langchain_db_url(conn_string: str) -> str:
    if conn_string.startswith("postgresql://"):
        return conn_string.replace("postgresql://", "postgresql+psycopg://", 1)
    if conn_string.startswith("postgres://"):
        return conn_string.replace("postgres://", "postgresql+psycopg://", 1)
    return conn_string


DB_URL = to_langchain_db_url(DB_DSN)
COLLECTION_NAME = "akasicdb_langchain_rag"

print(f"Collection: {COLLECTION_NAME}")
print("Database URL configured.")


## Step 3: Create the chat model and embedding model

RAG uses two different model calls:

- The embedding model converts document chunks and questions into vectors that can be compared by similarity.
- The chat model reads the retrieved text and writes the final answer.

This example uses `text-embedding-3-large` for embeddings and `gpt-4o-mini` for answer generation. You can swap either model as long as the replacement is supported by the corresponding LangChain integration.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

llm = init_chat_model("gpt-4o-mini", model_provider="openai")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


## Step 4: Load a source document

This cell fetches a public blog post and converts selected HTML content into a LangChain `Document` object.

We parse only the post title, header, and content so navigation menus, footers, scripts, and unrelated page text do not get embedded into the vector store. Cleaner source text usually produces better retrieval results.


In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

SOURCE_URL = "https://lilianweng.github.io/posts/2023-06-23-agent/"

response = requests.get(
    SOURCE_URL,
    timeout=20,
    headers={"User-Agent": "akasicdb-langchain-rag-example"},
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
selected = soup.find_all(class_=("post-content", "post-title", "post-header"))
text = " ".join(item.get_text(separator=" ", strip=True) for item in selected)
text = " ".join(text.split())

docs = [
    Document(
        page_content=text,
        metadata={"source": SOURCE_URL},
    )
]

print(f"Loaded {len(docs)} document(s).")
print(docs[0].metadata)
print(docs[0].page_content[:500])


## Step 5: Split the document into chunks

Embedding an entire long article as one vector usually makes retrieval too coarse. Splitting gives the retriever smaller units of meaning to search over.

`chunk_overlap` keeps a little context from the previous chunk, which helps when an idea crosses a chunk boundary. The exact values are application-dependent; smaller chunks improve precision, while larger chunks preserve more context.

LangChain provides text splitters for production use. Here we use a tiny local helper to keep the Colab environment simple and avoid unrelated dependency conflicts.


In [ ]:
def split_document(document: Document, chunk_size: int = 500, chunk_overlap: int = 200) -> list[Document]:
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    chunks = []
    text = document.page_content
    start = 0
    chunk_index = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        **document.metadata,
                        "chunk_index": chunk_index,
                    },
                )
            )
            chunk_index += 1

        start += chunk_size - chunk_overlap

    return chunks


all_splits = []
for doc in docs:
    all_splits.extend(split_document(doc, chunk_size=500, chunk_overlap=200))

print(f"Created {len(all_splits)} chunks.")
print(all_splits[0].page_content[:500])


## Step 6: Store embeddings in AkasicDB

`AkasicDBVector.from_documents()` embeds each chunk and writes both the text and vector representation into AkasicDB.

The `collection_name` groups related documents together. Re-running this cell with the same collection name may add another copy of the same chunks, depending on your vector store configuration. For experiments, changing `COLLECTION_NAME` is often the simplest way to start with a fresh collection.


In [ ]:
from langchain_akasicdb import AkasicDBVector

vector_store = AkasicDBVector.from_documents(
    documents=all_splits,
    embedding=embeddings,
    db_url=DB_URL,
    collection_name=COLLECTION_NAME,
)

print("Documents indexed in AkasicDB.")


## Step 7: Try retrieval by itself

Before connecting an LLM, it is useful to inspect what the vector store retrieves for a question. If the retrieved chunks are irrelevant, the final answer will usually be weak no matter which chat model you use.


In [ ]:
question = "What is Task Decomposition?"
retrieved_docs = vector_store.similarity_search(question, k=3)

for index, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Result {index} ---")
    print(f"Source: {doc.metadata.get('source')}")
    print(doc.page_content[:700])
    print()


## Step 8: Build a small RAG graph

The application has two steps:

1. `retrieve` searches AkasicDB for chunks related to the question.
2. `generate` sends those chunks to the chat model with instructions to answer from the provided context.

LangGraph keeps the steps explicit. For a small tutorial this may look slightly formal, but the graph structure becomes helpful when you later add routing, retries, query rewriting, or source validation.


In [ ]:
from typing import List

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import START, StateGraph
from typing_extensions import TypedDict

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an assistant for question-answering tasks. "
            "Use only the following retrieved context to answer the question. "
            "If the context does not contain the answer, say that you don't know. "
            "Keep the answer concise and cite the source URL when it is available.\n\n"
            "Context:\n{context}",
        ),
        ("human", "{question}"),
    ]
)


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


def retrieve(state: State):
    retrieved = vector_store.similarity_search(state["question"], k=4)
    return {"context": retrieved}


def generate(state: State):
    docs_content = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in state["context"]
    )
    messages = prompt.invoke(
        {"question": state["question"], "context": docs_content}
    )
    response = llm.invoke(messages)
    return {"answer": response.content}


graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()


## Step 9: Ask questions

The `ask()` helper runs the graph, prints the answer, and returns the full response object so you can inspect the retrieved context if needed.


In [ ]:
def ask(question: str):
    response = graph.invoke({"question": question})
    print(response["answer"])
    return response


response = ask("What is Task Decomposition?")


In [ ]:
response = ask("What is Chain of Hindsight?")


In [ ]:
response = ask("Does Chain of Hindsight add a regularization term?")


## Step 10: Inspect stored documents with `get()`

The LangChain AkasicDB vector store also provides a `get()` method for fetching stored records directly. This is useful for debugging metadata filters, checking whether ingestion worked, or previewing documents without doing a similarity search.

This notebook stores the original URL under the `source` metadata key.


In [ ]:
results = vector_store.get(
    limit=5,
    where={"source": SOURCE_URL},
    include=["documents", "metadatas"],
)

results


You can also filter by document text. The example below looks for chunks that contain a particular phrase.


In [ ]:
results = vector_store.get(
    where_document={"$contains": "Chain of Hindsight"},
    where={"source": SOURCE_URL},
    include=["documents", "metadatas"],
    limit=5,
)

results


## Recap

You now have a minimal RAG app that uses:

- LangChain document loaders and text splitters to prepare source material.
- OpenAI embeddings to convert text into vectors.
- AkasicDB to store and search those vectors.
- LangGraph to compose retrieval and generation into a simple application.

From here, common next steps are adding more URLs, attaching richer metadata, changing chunk sizes, evaluating retrieval quality, or exposing the `ask()` function through a small API or chat UI.
